# SAM 3.1 resistor dataset refinement + merge

This notebook has two independent parts:

1. **Single-image SAM 3.1 test first** — bootstrap SAM 3.1, upload one image, and inspect the result without downloading or processing the datasets.
2. **Full dataset refinement** — download the three COCO resistor datasets from the `resistor_model` GitHub `r1` release, compare/refine their labels with SAM 3.1, remove images with no accepted SAM annotation, deduplicate exact duplicates, and merge/split the result 80/10/10.

Use a GPU runtime. Visible progress is printed throughout the full workflow.

**Checkpoint:** this notebook uses the open `AEmotionStudio/sam3.1` mirror and downloads `sam3.1_multiplex.pt` (~3.5 GB) automatically. No Meta/Hugging Face access request is needed. The weights remain governed by the upstream SAM License. You can also set `SAM3_CHECKPOINT_PATH` to an existing local copy.

## 1. Bootstrap SAM 3.1 for the single-image test

Run this cell first. It installs the official SAM 3 code and downloads/loads the open `AEmotionStudio/sam3.1` multiplex checkpoint only; it does **not** download any resistor dataset.

In [ ]:
import urllib.request
print('[launcher] downloading SAM 3.1 single-image bootstrap...', flush=True)
url = 'https://raw.githubusercontent.com/Persie0/resistor_model/main/colab/sam3_test_bootstrap.py'
path = '/content/sam3_test_bootstrap.py'
urllib.request.urlretrieve(url, path)
exec(compile(open(path, encoding='utf-8').read(), path, 'exec'))


## 2. Test SAM 3.1 on one uploaded image

Upload one resistor image. This runs before the dataset workflow and uses the already loaded SAM 3.1 detector with the text prompt `resistor body`. It visualizes every returned mask with its score and bounding box.

In [ ]:
import io
import time
import matplotlib.pyplot as plt
from google.colab import files

if not globals().get('SAM3_BOOTSTRAPPED', False):
    raise RuntimeError('Run the SAM 3.1 bootstrap cell above first.')

print('[test 1/4] Upload one image...', flush=True)
uploaded_test = files.upload()
if not uploaded_test:
    raise RuntimeError('No image uploaded.')

test_name, test_bytes = next(iter(uploaded_test.items()))
print(f'[test 2/4] Loading {test_name} ({len(test_bytes) / 1024:.1f} KiB)...', flush=True)
test_image = Image.open(io.BytesIO(test_bytes)).convert('RGB')
print(f'[test] image size: {test_image.width}x{test_image.height}', flush=True)

print('[test 3/4] Running SAM 3.1 with prompt: resistor body', flush=True)
test_started = time.monotonic()
test_state = processor.set_image(test_image)
test_out = processor.set_text_prompt(prompt='resistor body', state=test_state)
test_elapsed = time.monotonic() - test_started

test_masks = test_out['masks'].squeeze(1).detach().cpu().numpy().astype(np.uint8)
test_boxes = test_out['boxes'].detach().cpu().numpy()
test_scores = test_out['scores'].detach().cpu().numpy().astype(float)

print(f'[test] inference complete in {test_elapsed:.2f}s | detections: {len(test_masks)}', flush=True)
for i, (box, score) in enumerate(zip(test_boxes, test_scores)):
    print(f'[test] detection {i}: score={score:.4f}, box={box.tolist()}', flush=True)

print('[test 4/4] Rendering result...', flush=True)
base = np.array(test_image).copy()
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(base)

for i, (mask, box, score) in enumerate(zip(test_masks, test_boxes, test_scores)):
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[..., 3] = mask.astype(np.float32) * 0.35
    overlay[..., 0] = mask.astype(np.float32)
    ax.imshow(overlay)

    x1, y1, x2, y2 = map(float, box)
    rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(x1, max(0, y1 - 5), f'{i}: {score:.3f}', fontsize=10, backgroundcolor='white')

ax.set_title(f'SAM 3.1 resistor-body detections: {test_name}')
ax.axis('off')
plt.show()
print('[test] done', flush=True)


## 3. Run the full three-dataset refinement/merge

Run this only after the bootstrap succeeded. The launcher reuses the exact `AEmotionStudio/sam3.1` checkpoint resolved by the bootstrap, skips the legacy authentication prompt in the older full-workflow script, then releases the single-image model before starting the full workflow. It downloads the three `r1` release datasets automatically and prints stage headers, download percent/speed, indexing counts, SAM image/instance progress with throughput/ETA, split information, and export progress.

In [ ]:
import gc
import os
import urllib.request
import huggingface_hub

if not globals().get('SAM3_BOOTSTRAPPED', False):
    raise RuntimeError('Run the SAM 3.1 bootstrap successfully before starting the full dataset workflow.')

sam3_checkpoint_for_full = globals().get('sam3_checkpoint')
if not sam3_checkpoint_for_full:
    raise RuntimeError('Bootstrap did not expose a resolved SAM 3.1 checkpoint path.')

# Preserve the open mirror checkpoint path for the full workflow.
os.environ['SAM3_CHECKPOINT_PATH'] = sam3_checkpoint_for_full

# The older full-workflow script still contains a Hugging Face auth stage. The
# mirror is public, so make that legacy prompt a no-op before the script imports it.
huggingface_hub.notebook_login = lambda *args, **kwargs: print('[auth] skipped: AEmotionStudio/sam3.1 is public', flush=True)
huggingface_hub.login = lambda *args, **kwargs: None

# Reuse the exact mirror checkpoint if the workflow asks the upstream builder for
# its default checkpoint. This prevents any request to facebook/sam3.
import sam3.model_builder as sam3_model_builder
sam3_model_builder.download_ckpt_from_hf = lambda version='sam3': sam3_checkpoint_for_full

print(f'[launcher] reusing AEmotionStudio SAM 3.1 checkpoint: {sam3_checkpoint_for_full}', flush=True)
print('[launcher] releasing single-image SAM 3.1 model before full workflow...', flush=True)
try:
    del test_state
except NameError:
    pass
del processor
del model
SAM3_BOOTSTRAPPED = False
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('[launcher] downloading latest SAM merge workflow...', flush=True)
url = 'https://raw.githubusercontent.com/Persie0/resistor_model/main/colab/sam3_merge_3_resistor_datasets.py'
path = '/content/sam3_merge_3_resistor_datasets.py'
urllib.request.urlretrieve(url, path)
print('[launcher] starting full dataset workflow with reused SAM 3.1 checkpoint...', flush=True)
exec(compile(open(path, encoding='utf-8').read(), path, 'exec'))
